
# <font color = "blue"> tutorial from https://debuggercafe.com/text-generation-with-transformers/#download-code

** added temperature argument for inference

# imports

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from collections import Counter

# dataset preparation

In [9]:
# Dataset Preparation
with open('data/alice_1.txt', 'r', encoding='utf-8') as file:
    text = file.read()
# Tokenize the text into words
words = text.split()
word_counts = Counter(words)
vocab = list(word_counts.keys())
vocab_size = len(vocab)
word_to_int = {word: i for i, word in enumerate(vocab)}
int_to_word = {i: word for word, i in word_to_int.items()}
SEQUENCE_LENGTH = 64
samples = [words[i:i+SEQUENCE_LENGTH+1] for i in range(len(words)-SEQUENCE_LENGTH)]
print(vocab)
print(word_to_int)
print(int_to_word)

['Alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by', 'her', 'sister', 'on', 'the', 'bank,', 'and', 'having', 'nothing', 'do:', 'once', 'or', 'twice', 'she', 'had', 'peeped', 'into', 'book', 'reading,', 'but', 'it', 'no', 'pictures', 'conversations', 'in', 'it,', '`and', 'what', 'is', 'use', 'a', "book,'", 'thought', '`without', "conversation?'", 'So', 'considering', 'own', 'mind', '(as', 'well', 'as', 'could,', 'for', 'hot', 'day', 'made', 'feel', 'sleepy', 'stupid),', 'whether', 'pleasure', 'making', 'daisy-chain', 'would', 'be', 'worth', 'trouble', 'getting', 'up', 'picking', 'daisies,', 'when', 'suddenly', 'White', 'Rabbit', 'with', 'pink', 'eyes', 'ran', 'close', 'her.', 'There', 'so', 'VERY', 'remarkable', 'that;', 'nor', 'did', 'think', 'much', 'out', 'way', 'hear', 'say', 'itself,', '`Oh', 'dear!', 'Oh', 'I', 'shall', "late!'", '(when', 'over', 'afterwards,', 'occurred', 'that', 'ought', 'have', 'wondered', 'at', 'this,', 'time', 'all', 'seemed', 'q

# creating dataset class and data loader

In [10]:
class TextDataset(Dataset):
    def __init__(self, samples, word_to_int):
        self.samples = samples
        self.word_to_int = word_to_int
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        sample = self.samples[idx]
        input_seq = torch.LongTensor([self.word_to_int[word] for word in sample[:-1]])
        target_seq = torch.LongTensor([self.word_to_int[word] for word in sample[1:]])
        return input_seq, target_seq

In [11]:
BATCH_SIZE = 32
dataset = TextDataset(samples, word_to_int)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)
print(dataset[1])

(tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  7, 16, 17,
         3, 18, 19, 20, 21, 22, 23, 24, 25, 13, 26, 10, 11,  1, 27, 28, 29, 23,
        30, 31, 20, 32, 33, 34, 35, 36, 37, 13, 38,  7, 39, 40, 41,  0, 42, 31,
        20, 43, 44, 22,  1, 45, 33, 10, 46, 47]), tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  7, 16, 17,  3,
        18, 19, 20, 21, 22, 23, 24, 25, 13, 26, 10, 11,  1, 27, 28, 29, 23, 30,
        31, 20, 32, 33, 34, 35, 36, 37, 13, 38,  7, 39, 40, 41,  0, 42, 31, 20,
        43, 44, 22,  1, 45, 33, 10, 46, 47, 48]))


# causal masking and positional encoding methods

In [12]:
#CAUsAL MASKING
def generate_square_subsequent_mask(sz):
  """
  Generate a square mask for the sequence. The masked positions are filled with float('-inf').
  Unmasked positions are filled with float(0.0).
  """
  mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
  mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
  return mask

In [13]:
# POSITIONAL ENCODING
class PositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model, dropout=0.1):
        """
        :param max_len: Input length sequence.
        :param d_model: Embedding dimension.
        :param dropout: Dropout value (default=0.1)
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self, x):
        """
        Inputs of forward function
        :param x: the sequence fed to the positional encoder model (required).
        Shape:
            x: [sequence length, batch size, embed dim]
            output: [sequence length, batch size, embed dim]
        """
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# decoder-only transformer model

In [22]:
class TextGen(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_layers, num_heads):
        super(TextGen, self).__init__()
        self.pos_encoder = PositionalEncoding(max_len=SEQUENCE_LENGTH, d_model=embed_dim)
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer=self.decoder_layer,
            num_layers=num_layers,
        )
        self.linear = nn.Linear(embed_dim, vocab_size)
        self.dropout = nn.Dropout(0.3) # originally 0.2

    # Positional encoding is required. Else the model does not learn.
    def forward(self, x):
        emb = self.emb(x)

        # Generate input sequence mask with shape (SEQUENCE_LENGTH, SEQUENCE_LENGTH)
        input_mask = generate_square_subsequent_mask(x.size(1)).to(x.device)

        x = self.pos_encoder(emb)
        x = self.decoder(x, memory=x, tgt_mask=input_mask, memory_mask=input_mask)
        x = self.dropout(x)
        out = self.linear(x)
        return out

# training - hyperparameters & initialize model

In [28]:
epochs = 10 # temp epochsrhgug
learning_rate = 0.001
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TextGen(
    vocab_size=vocab_size,
    embed_dim=100,
    num_layers=2,
    num_heads=2,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
print(model)
# Total parameters and trainable parameters.
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")
total_trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.\n")

TextGen(
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (emb): Embedding(5297, 100)
  (decoder_layer): TransformerDecoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
    )
    (multihead_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
    )
    (linear1): Linear(in_features=100, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=100, bias=True)
    (norm1): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
    (norm3): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
    (dropout3): Dropout(p=0.1, inplace=False)
  )
  (decoder): Transform

# training method

In [29]:
# Training
def train(model, epochs, dataloader, criterion):
    model.train()
    for epoch in range(epochs):
        running_loss = 0
        for input_seq, target_seq in dataloader:
            input_seq, target_seq = input_seq.to(device), target_seq.to(device)
            outputs = model(input_seq)
            target_seq = target_seq.contiguous().view(-1)
            outputs = outputs.view(-1, vocab_size)

            loss = criterion(outputs, target_seq.view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.detach().cpu().numpy()
        epoch_loss = running_loss / len(dataloader)
        print(f"Epoch {epoch} loss: {epoch_loss:.3f}")




In [26]:
train(model, epochs, dataloader, criterion)

Epoch 0 loss: 4.351
Epoch 1 loss: 2.038
Epoch 2 loss: 1.309
Epoch 3 loss: 0.996
Epoch 4 loss: 0.823
Epoch 5 loss: 0.712
Epoch 6 loss: 0.633
Epoch 7 loss: 0.568
Epoch 8 loss: 0.513
Epoch 9 loss: 0.476


"One thing to note in the training loop is the shape of the targets and the outputs before calculating the loss. We need to ensure that the shape of the targets is [batch_size x sequence_length] in flattened format and the shape of the outputs is [batch_size x sequence_length, vocab_size]."

# inference

In [33]:
def return_int_vector(text):
    words = text.split()
    input_seq = torch.LongTensor([word_to_int[word] for word in words[-SEQUENCE_LENGTH:]]).unsqueeze(0)
    return input_seq

def sample_next(predictions, temperature=1.0, top_k=None):
    """
    Sample the next token using temperature and top-k sampling.

    :param predictions: Model logits for the next word.
    :param temperature: Controls randomness (higher = more random).
    :param top_k: If set, restricts sampling to top-k most likely words.
    """
    probabilities = F.softmax(predictions[:, -1, :] / temperature, dim=-1).cpu()

    if top_k is not None:
        # Select top-k probabilities
        top_values, top_indices = torch.topk(probabilities, top_k)
        probabilities = top_values / torch.sum(top_values)  # Re-normalize
        next_token = torch.multinomial(probabilities, 1).item()
        next_token = top_indices[next_token].item()  # Convert to actual token index
    else:
        # Sample from full distribution
        next_token = torch.multinomial(probabilities, 1).item()

    return next_token

def text_generator(sentence, generate_length, temperature=1.0, top_k=None):
    model.eval()
    sample = sentence
    for i in range(generate_length):
        int_vector = return_int_vector(sample)
        if len(int_vector) >= SEQUENCE_LENGTH - 1:
            break
        input_tensor = int_vector.to(device)
        with torch.no_grad():
            predictions = model(input_tensor)
        next_token = sample_next(predictions, temperature, top_k)
        sample += ' ' + int_to_word[next_token]
    print(sample)
    print('\n')

# sample inference

In [37]:
sentences = ["Alice was sleepy"]
generate_length = 100
temperature = 1.0
for sentence in sentences:
    print(f"PROMPT: {sentence}")
    temperature = 1.0
    print("TEMPERATURE " + str(temperature) + ":")
    text_generator(sentence, generate_length, temperature)
    temperature = 0.5
    print("TEMPERATURE " + str(temperature) + ":")
    text_generator(sentence, generate_length, temperature)
    temperature = 0.2
    print("TEMPERATURE " + str(temperature) + ":")
    text_generator(sentence, generate_length, temperature)
#

PROMPT: Alice was sleepy
TEMPERATURE 1.0:
Alice was sleepy washing?' pitied door-- this. round, dead late, toes?' Cat's disobey, absurd rose surprise it?' grave `How him, sell two any `What! ground appeared bread-and-butter, asking temper,' laid ordered `Tell `There ran, running other Ann! unimportant--important--' They're over!' fact, aloud; sure delighted BEST hot, alone. ancient denied `Back pretexts WASHING--extra."' peeping already creature, straightened `Nothing from hurt, different,' last!' hearts. turned around, partners--' confusion story.' `Come no joys, introduced slates'll pope, YOU. consented creatures together, decidedly hope Dormouse,' * (as executioner, too.' course-- change Hare,) fairly,' sadly. cut temper,' jaws along rats `W. treacle-well.' leaves, Wonderland pretexts well scolded said; `By-the-bye,


TEMPERATURE 0.5:
Alice was sleepy sounds mean conversation?' peeped words.' Hatter. violently, whiting. tears! liked pressed `Each to--' lessons!' spoke door, "Let gue